In [2]:
import wfdb
import numpy as np
import pandas as pd
import os

In [3]:
# File paths (without extensions for wfdb)
data_path = '/Users/ajaybirheir/Documents/GitHub/ICPreg/Data/charis1'

# Read the record
record = wfdb.rdrecord(data_path)

# Check metadata
print("Signal names:", record.sig_name)
print("Number of samples:", record.sig_len)
print("Sampling frequency:", record.fs)

# Get signal data as a NumPy array
signals = record.p_signal  # shape: (samples, channels)
print("Signals shape:", signals.shape)

# Create a labeled table for the ABP, ECG, and ICP channels.
signals_df = pd.DataFrame(signals, columns=record.sig_name)
signals_abp_ecg_icp = signals_df[["ABP", "ECG", "ICP"]]



Signal names: ['ABP', 'ECG', 'ICP']
Number of samples: 12239851
Sampling frequency: 50
Signals shape: (12239851, 3)


In [4]:
print(signals_abp_ecg_icp.head())

        ABP       ECG        ICP
0 -0.011634  0.078763  21.079613
1 -0.011634  0.063800  20.573703
2 -0.046537  0.031078  20.468305
3  0.279222  0.014963  20.963676
4  0.162880  0.002138  20.984755


In [5]:
window_size = 15000
max_datapoint = 4320000

# Keep datapoints 1 through 4,320,000 (Python index 0 through 4,319,999).
window_data = signals_df.loc[: max_datapoint - 1, ["ICP", "ABP", "ECG"]].copy()
window_data["window_number"] = np.arange(len(window_data)) // window_size + 1

window_summary = (
    window_data.groupby("window_number")[["ICP", "ABP", "ECG"]]
    .agg(["mean", "max", "std"])
    .reset_index()
)

window_summary.columns = [
    "window_number",
    "icp_mean", "icp_max", "icp_std",
    "abp_mean", "abp_max", "abp_std",
    "ecg_mean", "ecg_max", "ecg_std",
]

window_summary.insert(1, "start_datapoint", (window_summary["window_number"] - 1) * window_size + 1)
window_summary.insert(2, "end_datapoint", window_summary["window_number"] * window_size)

window_summary.head()


   window_number  start_datapoint  end_datapoint   icp_mean    icp_max   icp_std   abp_mean     abp_max    abp_std  ecg_mean   ecg_max   ecg_std
0              1                1         15000  15.507324  28.660002  3.175714  81.384198  171.127079  14.454426  0.002419  1.191502  0.288351
1              2            15001         30000  17.777943  24.188907  1.780462  83.421651  170.661514  17.700970  0.001082  1.068811  0.271393
2              3            30001         45000  12.860375  18.915072  2.565671  84.031683  175.548409  17.096859  0.001221  1.278233  0.287677
3              4            45001         60000   7.269542  14.355870  2.344191  83.179784  180.317207  11.729725  0.001359  1.033908  0.274615
4              5            60001         75000   7.705490  14.929268  1.702141  84.247033  168.917133  14.069663  0.001449  1.112449  0.281935